<a href="https://colab.research.google.com/github/bhavyajaiswal27/SummerDB/blob/master/Fertilizer_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!mkdir -p ~/.kaggle
!cp '/content/kaggle (1).json' ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [8]:
!kaggle competitions download -c playground-series-s5e6

  0% 0.00/11.7M [00:00<?, ?B/s]
100% 11.7M/11.7M [00:00<00:00, 1.14GB/s]


In [9]:
!unzip playground-series-s5e6.zip -d playground-series-s5e6

Archive:  playground-series-s5e6.zip
  inflating: playground-series-s5e6/sample_submission.csv  
  inflating: playground-series-s5e6/test.csv  
  inflating: playground-series-s5e6/train.csv  


In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.0 MB/s eta 0:00:00


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [10]:
train = pd.read_csv('/content/playground-series-s5e6/train.csv')
test = pd.read_csv('/content/playground-series-s5e6/test.csv')

In [11]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [12]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               750000 non-null  int64 
 1   Temparature      750000 non-null  int64 
 2   Humidity         750000 non-null  int64 
 3   Moisture         750000 non-null  int64 
 4   Soil Type        750000 non-null  object
 5   Crop Type        750000 non-null  object
 6   Nitrogen         750000 non-null  int64 
 7   Potassium        750000 non-null  int64 
 8   Phosphorous      750000 non-null  int64 
 9   Fertilizer Name  750000 non-null  object
dtypes: int64(7), object(3)
memory usage: 57.2+ MB


In [14]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id           250000 non-null  int64 
 1   Temparature  250000 non-null  int64 
 2   Humidity     250000 non-null  int64 
 3   Moisture     250000 non-null  int64 
 4   Soil Type    250000 non-null  object
 5   Crop Type    250000 non-null  object
 6   Nitrogen     250000 non-null  int64 
 7   Potassium    250000 non-null  int64 
 8   Phosphorous  250000 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 17.2+ MB


In [15]:
x = train.drop(['id', 'Fertilizer Name'], axis=1)
y = train['Fertilizer Name']

In [17]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [18]:
def quantile_bin_encode(df, cols, q=5, labels=['very low', 'low', 'medium', 'high', 'very high']):
    df_transformed = df.copy()

    for col in cols:
        binned = pd.qcut(df_transformed[col], q=q, labels=labels)

        label_map = {label: idx for idx, label in enumerate(labels)}
        df_transformed[f"{col}_bin"] = binned.map(label_map).astype('int64')

    return df_transformed

In [21]:
numerical_cols = x.select_dtypes(include=['int64', 'float64']).columns
X_binned = quantile_bin_encode(x, numerical_cols)

X_binned.head()


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Temparature_bin,Humidity_bin,Moisture_bin,Nitrogen_bin,Potassium_bin,Phosphorous_bin
0,37,70,36,Clayey,Sugarcane,36,4,5,4,4,1,4,1,0
1,27,69,65,Sandy,Millets,30,6,18,0,4,4,3,1,2
2,29,63,32,Sandy,Millets,24,12,16,1,2,0,2,3,1
3,35,62,54,Sandy,Barley,39,12,4,3,2,3,4,3,0
4,35,58,43,Red,Paddy,37,2,16,3,1,2,4,0,1


In [22]:
test_binned = quantile_bin_encode(test, numerical_cols)
test_binned.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Temparature_bin,Humidity_bin,Moisture_bin,Nitrogen_bin,Potassium_bin,Phosphorous_bin
0,750000,31,70,52,Sandy,Wheat,34,11,24,2,4,3,3,2,2
1,750001,27,62,45,Red,Sugarcane,30,14,15,0,2,2,3,3,1
2,750002,28,72,28,Clayey,Ground Nuts,14,15,4,1,4,0,1,3,0
3,750003,37,53,57,Black,Ground Nuts,18,17,36,4,0,3,1,4,4
4,750004,31,55,32,Red,Pulses,13,19,14,2,1,0,1,4,1


In [24]:
original = pd.read_csv('/content/Fertilizer Prediction.csv')
original.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,32,51,41,Red,Ground Nuts,7,3,19,14-35-14
1,35,58,35,Black,Cotton,4,14,16,Urea
2,27,55,43,Sandy,Sugarcane,28,0,17,20-20
3,33,56,56,Loamy,Ground Nuts,37,5,24,28-28
4,32,70,60,Red,Ground Nuts,4,6,9,14-35-14


In [25]:
orig_copy = original.copy()

# Number of copies
n = 6
for i in range(n):
    original = pd.concat([original, orig_copy], axis=0, ignore_index=True)

original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Temparature      700000 non-null  int64 
 1   Humidity         700000 non-null  int64 
 2   Moisture         700000 non-null  int64 
 3   Soil Type        700000 non-null  object
 4   Crop Type        700000 non-null  object
 5   Nitrogen         700000 non-null  int64 
 6   Potassium        700000 non-null  int64 
 7   Phosphorous      700000 non-null  int64 
 8   Fertilizer Name  700000 non-null  object
dtypes: int64(6), object(3)
memory usage: 48.1+ MB


In [26]:
original_binned = quantile_bin_encode(original, numerical_cols)

original_binned.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name,Temparature_bin,Humidity_bin,Moisture_bin,Nitrogen_bin,Potassium_bin,Phosphorous_bin
0,32,51,41,Red,Ground Nuts,7,3,19,14-35-14,2,0,1,0,0,2
1,35,58,35,Black,Cotton,4,14,16,Urea,3,1,1,0,3,1
2,27,55,43,Sandy,Sugarcane,28,0,17,20-20,0,1,2,3,0,1
3,33,56,56,Loamy,Ground Nuts,37,5,24,28-28,2,1,3,4,1,2
4,32,70,60,Red,Ground Nuts,4,6,9,14-35-14,2,4,4,0,1,1


In [28]:
from sklearn.metrics import classification_report

In [29]:
# Store scores
f1_scores = []
map3_scores = []
models = []

# Collect predictions and true labels across all folds
all_y_true = []
all_y_pred = []

# Prepare K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_binned, y_encoded)):
    print(f"\n***** Fold {fold + 1} *****")

    # Make full copies to avoid warnings
    X_train = X_binned.iloc[train_idx].copy()
    X_val = X_binned.iloc[val_idx].copy()
    y_train = y_encoded[train_idx]
    y_val = y_encoded[val_idx]

    # Combine original with train data
    X_train = pd.concat([X_train, original_binned], ignore_index=True)
    y_train = np.concatenate([y_train, le.transform(original_binned['Fertilizer Name'])])

    # Drop target column from training data
    X_train.drop(columns=['Fertilizer Name'], inplace=True)

    #cols_to_category = list(X_train.select_dtypes(include=['object']).columns) + ['A', 'B', 'C']

    # Convert all selected features to categorical
    for col in X_train.select_dtypes(include='object').columns:
        X_train[col] = X_train[col].astype('category')

    for col in X_val.select_dtypes(include='object').columns:
        X_val[col] = X_val[col].astype('category')

    cat_features = X_train.select_dtypes(include='category').columns.tolist()

    # For debugging purposes
    # print(cat_features)
    # print(X_train.info())
    # print(X_val.info())

    model = CatBoostClassifier(
        iterations=10000,
        depth=6,
        learning_rate=0.03,
        early_stopping_rounds=100,
        task_type="GPU",
        loss_function="MultiClass",
        eval_metric="MultiClass",
        l2_leaf_reg=0.86,
        bootstrap_type="Bayesian",
        bagging_temperature=0.5,
        random_strength=2.65,
        border_count=124,
        verbose=500
    )

    model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val))

    # Predict class labels and probabilities
    y_pred = model.predict(X_val)
    y_probs = model.predict_proba(X_val)

    # Store predictions and true labels
    all_y_true.extend(y_val)
    all_y_pred.extend(y_pred)

    # F1 Score
    report = classification_report(y_val, y_pred, output_dict=True)
    f1_macro = report["macro avg"]["f1-score"]
    f1_scores.append(f1_macro)

    # MAP@3
    top3_preds = np.argsort(y_probs, axis=1)[:, -3:][:, ::-1]

    def mapk(actual, predicted, k=3):
        def apk(a, p, k):
            if a in p[:k]:
                return 1.0 / (p[:k].index(a) + 1)
            return 0.0
        return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

    map3 = mapk(y_val.tolist(), top3_preds.tolist(), k=3)
    map3_scores.append(map3)
    models.append(model)

    print(f"F1 (macro): {f1_macro:.4f} | MAP@3: {map3:.4f}")

# Final Results
print("\n***** Final CV Results *****")
print(f"Avg F1: {np.mean(f1_scores):.4f}")
print(f"Avg MAP@3: {np.mean(map3_scores):.4f}")


***** Fold 1 *****
0:	learn: 1.9457896	test: 1.9457785	best: 1.9457785 (0)	total: 66.1ms	remaining: 11m 1s
500:	learn: 1.9253429	test: 1.9339890	best: 1.9339890 (500)	total: 24.6s	remaining: 7m 47s
1000:	learn: 1.9094844	test: 1.9291231	best: 1.9291231 (1000)	total: 50.7s	remaining: 7m 35s
1500:	learn: 1.8961996	test: 1.9260877	best: 1.9260877 (1500)	total: 1m 16s	remaining: 7m 13s
2000:	learn: 1.8838229	test: 1.9237525	best: 1.9237525 (2000)	total: 1m 42s	remaining: 6m 50s
2500:	learn: 1.8723652	test: 1.9219810	best: 1.9219810 (2500)	total: 2m 10s	remaining: 6m 29s
3000:	learn: 1.8615260	test: 1.9207519	best: 1.9207519 (3000)	total: 2m 36s	remaining: 6m 4s
3500:	learn: 1.8512265	test: 1.9197681	best: 1.9197681 (3500)	total: 3m 2s	remaining: 5m 39s
4000:	learn: 1.8411923	test: 1.9189288	best: 1.9189288 (4000)	total: 3m 29s	remaining: 5m 13s
4500:	learn: 1.8319800	test: 1.9183758	best: 1.9183725 (4496)	total: 3m 55s	remaining: 4m 48s
5000:	learn: 1.8227996	test: 1.9179523	best: 1.91795

In [30]:
# Convert test data
for col in cat_features:
    test_binned[col] = test_binned[col].astype('category')

# Accumulate prediction probabilities
all_preds = np.zeros((test_binned.shape[0], len(le.classes_)))

X_test = test_binned.drop(columns='id')

for model in models:
    probs = model.predict_proba(X_test)
    all_preds += probs

# Average over folds
avg_preds = all_preds / len(models)

# Get top 3 indices like before
top3_preds = np.argsort(probs, axis=1)[:, -3:][:, ::-1]  # Top 3 class indices, descending order

# Convert class indices back to original label strings
top3_labels = le.inverse_transform(top3_preds.ravel()).reshape(top3_preds.shape)

submission = pd.DataFrame({
    'id': test['id'],  # Replace with actual ID column name
    'Fertilizer Name': [' '.join(row) for row in top3_labels]
})

submission.to_csv('submission4.csv', index=False)
print("Done!")


Done!


## **Loda**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/content/playground-series-s5e6/train.csv')
test = pd.read_csv('/content/playground-series-s5e6/test.csv')

   id  Temparature  Humidity  Moisture Soil Type  Crop Type  Nitrogen  \
0   0           37        70        36    Clayey  Sugarcane        36   
1   1           27        69        65     Sandy    Millets        30   
2   2           29        63        32     Sandy    Millets        24   
3   3           35        62        54     Sandy     Barley        39   
4   4           35        58        43       Red      Paddy        37   

   Potassium  Phosphorous Fertilizer Name  
0          4            5           28-28  
1          6           18           28-28  
2         12           16        17-17-17  
3         12            4        10-26-26  
4          2           16             DAP  
       id  Temparature  Humidity  Moisture Soil Type    Crop Type  Nitrogen  \
0  750000           31        70        52     Sandy        Wheat        34   
1  750001           27        62        45       Red    Sugarcane        30   
2  750002           28        72        28    Clayey  Ground

In [ ]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [ ]:
test.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,750000,31,70,52,Sandy,Wheat,34,11,24
1,750001,27,62,45,Red,Sugarcane,30,14,15
2,750002,28,72,28,Clayey,Ground Nuts,14,15,4
3,750003,37,53,57,Black,Ground Nuts,18,17,36
4,750004,31,55,32,Red,Pulses,13,19,14


In [ ]:
train.columns

Index(['id', 'Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name'],
      dtype='object')

In [ ]:
from sklearn.preprocessing import LabelEncoder
ll = LabelEncoder()
mm = LabelEncoder()
train['Soil Type'] = ll.fit_transform(train['Soil Type'])
train['Crop Type'] = ll.fit_transform(train['Crop Type'])
train['Fertilizer Name'] = mm.fit_transform(train['Fertilizer Name'])
test['Soil Type'] = ll.fit_transform(test['Soil Type'])
test['Crop Type'] = ll.fit_transform(test['Crop Type'])

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
col = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
train[col] = scaler.fit_transform(train[col])
test[col] = scaler.fit_transform(test[col])

In [ ]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,0.923077,0.909091,0.275,1,8,0.842105,0.210526,0.119048,4
1,1,0.153846,0.863636,1.000,4,4,0.684211,0.315789,0.428571,4
2,2,0.307692,0.590909,0.175,4,4,0.526316,0.631579,0.380952,2
3,3,0.769231,0.545455,0.725,4,0,0.921053,0.631579,0.095238,0
4,4,0.769231,0.363636,0.450,3,6,0.868421,0.105263,0.380952,5


In [ ]:
test.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,750000,0.461538,0.909091,0.675,4,10,0.789474,0.578947,0.571429
1,750001,0.153846,0.545455,0.500,3,8,0.684211,0.736842,0.357143
2,750002,0.230769,1.000000,0.075,1,2,0.263158,0.789474,0.095238
3,750003,0.923077,0.136364,0.800,0,2,0.368421,0.894737,0.857143
4,750004,0.461538,0.227273,0.175,3,7,0.236842,1.000000,0.333333


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(train.drop(['id', 'Fertilizer Name'], axis=1), train['Fertilizer Name'])

LinearRegression()

In [ ]:
ypred = model.predict(test.drop(['id'], axis=1))

In [ ]:
print(ypred.max)

<built-in method max of numpy.ndarray object at 0x7fc64223b8d0>


In [ ]:
ypred = mm.inverse_transform(ypred.astype(int))

In [ ]:
submit = pd.DataFrame({'id': test['id'], 'Fertilizer Name': ypred})
submit.to_csv('submission.csv', index=False)

# Rejected Part

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/playground-series-s5e6/train.csv')
df.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               750000 non-null  int64 
 1   Temparature      750000 non-null  int64 
 2   Humidity         750000 non-null  int64 
 3   Moisture         750000 non-null  int64 
 4   Soil Type        750000 non-null  object
 5   Crop Type        750000 non-null  object
 6   Nitrogen         750000 non-null  int64 
 7   Potassium        750000 non-null  int64 
 8   Phosphorous      750000 non-null  int64 
 9   Fertilizer Name  750000 non-null  object
dtypes: int64(7), object(3)
memory usage: 57.2+ MB


In [ ]:
df.shape

(750000, 10)

In [ ]:
df.isnull().any()

,0
id,False
Temparature,False
Humidity,False
Moisture,False
Soil Type,False
Crop Type,False
Nitrogen,False
Potassium,False
Phosphorous,False
Fertilizer Name,False


In [ ]:
df.columns

Index(['id', 'Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name'],
      dtype='object')

In [ ]:
df['Fertilizer Name'].unique()

array(['28-28', '17-17-17', '10-26-26', 'DAP', '20-20', '14-35-14',
       'Urea'], dtype=object)

In [ ]:
df.describe()

,id,Temparature,Humidity,Moisture,Nitrogen,Potassium,Phosphorous
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,31.503565,61.038912,45.184147,23.093808,9.478296,21.073227
std,216506.495284,4.025574,6.647695,11.794594,11.216125,5.765622,12.346831
min,0.000000,25.000000,50.000000,25.000000,4.000000,0.000000,0.000000
25%,187499.750000,28.000000,55.000000,35.000000,13.000000,4.000000,10.000000
50%,374999.500000,32.000000,61.000000,45.000000,23.000000,9.000000,21.000000
75%,562499.250000,35.000000,67.000000,55.000000,33.000000,14.000000,32.000000
max,749999.000000,38.000000,72.000000,65.000000,42.000000,19.000000,42.000000


In [ ]:
coll = ['Soil Type', 'Crop Type', 'Fertilizer Name']
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for i in coll:
  df[i] = le.fit_transform(df[i])

In [ ]:
col = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[col] = scaler.fit_transform(df[col])


In [ ]:
df.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,0.923077,0.909091,0.275,1,8,0.842105,0.210526,0.119048,4
1,1,0.153846,0.863636,1.000,4,4,0.684211,0.315789,0.428571,4
2,2,0.307692,0.590909,0.175,4,4,0.526316,0.631579,0.380952,2
3,3,0.769231,0.545455,0.725,4,0,0.921053,0.631579,0.095238,0
4,4,0.769231,0.363636,0.450,3,6,0.868421,0.105263,0.380952,5


In [ ]:
df['Crop Type'].unique()

array([ 8,  4,  0,  6,  7,  9,  2,  3,  1, 10,  5])

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(df.drop(['id', 'Fertilizer Name'], axis=1), df['Fertilizer Name'])

LinearRegression()

In [ ]:
test = pd.read_csv('/content/playground-series-s5e6/test.csv')
test.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,750000,31,70,52,Sandy,Wheat,34,11,24
1,750001,27,62,45,Red,Sugarcane,30,14,15
2,750002,28,72,28,Clayey,Ground Nuts,14,15,4
3,750003,37,53,57,Black,Ground Nuts,18,17,36
4,750004,31,55,32,Red,Pulses,13,19,14


In [ ]:
test.isnull().any()

,0
id,False
Temparature,False
Humidity,False
Moisture,False
Soil Type,False
Crop Type,False
Nitrogen,False
Potassium,False
Phosphorous,False


In [ ]:
coll = ['Soil Type', 'Crop Type']
for i in coll:
  test[i] = le.fit_transform(test[i])

In [ ]:
col = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
test[col] = scaler.fit_transform(test[col])

In [ ]:
test.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,750000,0.461538,0.909091,0.675,4,10,0.789474,0.578947,0.571429
1,750001,0.153846,0.545455,0.500,3,8,0.684211,0.736842,0.357143
2,750002,0.230769,1.000000,0.075,1,2,0.263158,0.789474,0.095238
3,750003,0.923077,0.136364,0.800,0,2,0.368421,0.894737,0.857143
4,750004,0.461538,0.227273,0.175,3,7,0.236842,1.000000,0.333333


In [ ]:
test.columns

Index(['id', 'Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous'],
      dtype='object')

In [ ]:
y_pred = model.predict(test.drop(['id'], axis=1))

In [ ]:
y_pred = le.inverse_transform(y_pred.astype(int))

In [ ]:
submit = pd.DataFrame({'id': test['id'], 'Fertilizer Name': y_pred})
submit.to_csv('submission.csv', index=False)

In [ ]:
!kaggle competitions submit -c <playground-series-s5e6> -f submission.csv -m "First submission"